In [ ]:
import pandas as pd
import numpy as np
from typing import NamedTuple

In [ ]:
# 1.数据加载
data = pd.read_excel("./TRY1.xlsx")
data = data.loc[:,["交易日期","市盈率TTM","单位净值"]]
data.rename(columns={"交易日期":"Date","市盈率TTM":"PE","单位净值":"Net"},inplace=True)
data.index = data["Date"].astype("datetime64[D]")

In [ ]:
# 2.交易参数设置
class TradingConfig(NamedTuple):
    # 第一阈值PE及其对应投资金额
    entry_threshold_pe_1: float = 13
    investment_amount_1: int = 200
    
    # 第二阈值PE及其对应投资金额
    entry_threshold_pe_2: float = 15
    investment_amount_2: int = 100
    
    # date
    start_date:str = "2024-01-01"
    end_date:str = "2024-12-30"
config = TradingConfig()
data = data[(data.index>= config.start_date) & (data.index<=config.end_date)]

In [ ]:
data["DailyReturn"] = (data["Net"] / data["Net"].shift(1) - 1).fillna(1)
buy_order = np.zeros_like(data["DailyReturn"],dtype=np.float64)
data["PE"] =data["PE"].shift(1).fillna(20)

In [ ]:
threshold_pe_1_flag = data["PE"]< config.entry_threshold_pe_1
threshold_pe_2_flag = (data["PE"]>=config.entry_threshold_pe_1)&(data["PE"]<config.entry_threshold_pe_2)
close_pe_flag = data["PE"]> config.entry_threshold_pe_2

buy_order[threshold_pe_1_flag] = 200 / data["Net"][threshold_pe_1_flag]
buy_order[threshold_pe_2_flag] = 100  / data["Net"][threshold_pe_2_flag]
buy_order_cumsum = np.cumsum(buy_order)
sell_order_cumsum = np.zeros_like(data["DailyReturn"],dtype=np.int64)
sell_order = np.zeros_like(data["DailyReturn"],dtype=np.int64)
sell_order_cumsum[close_pe_flag] = -buy_order_cumsum[close_pe_flag]
shift_sell_order_cumsum = np.roll(-buy_order_cumsum[close_pe_flag],shift=1)
shift_sell_order_cumsum[0] = 0
sell_order[close_pe_flag] = sell_order_cumsum[close_pe_flag] - shift_sell_order_cumsum
order = buy_order.copy()
order[close_pe_flag] = sell_order[close_pe_flag]

In [ ]:
data["Positions"] = order.cumsum()
data["DailyReturnShift"] = np.roll(data["DailyReturn"],shift=-1)
data["DailyReturnShift"][-1] = 0
data["Order"] = order
buy_flag = np.cumsum(order * data["Net"]) > 0
buy_cash = buy_order.copy() * data["Net"]
buy_cash[~buy_flag] = 0

In [ ]:
data["PnL"] = np.cumsum(data["Positions"] * data["DailyReturnShift"])
data["Cash"] = buy_cash.cumsum()

In [ ]:
# Positions
pd.Series(data["Positions"]).plot()

In [ ]:
# PnL ratio
pd.Series(data["PnL"] /data["Cash"]).plot()

In [ ]:
# PnL
pd.Series(data["PnL"]).plot()

In [ ]:
data

20.4 市盈率 清盘
[13.6,20.4) 买入100
(0,13.6) 买入200